# Study 923 — The Cash Lag — the teardown

Window-free duration regressions, the distributed-lag pass-through profile, the proxy-window artefact sweep, the full pre-registered lookback grid, the cost sweep, the era cut, the block-bootstrap CI, the SGOV cross-check, and a live three-world synthetic control. Every real-tape number is frozen from `docs/results.md` (Fingerprint `d78f3661ba09`); the live cells are synthetic and labelled as such.

**Sample.** BIL ∩ USFR ∩ SHV ∩ ^IRX, 2014-02-04 → 2026-06-30, n = 3,118, daily total-return closes (`auto_adjust=True`); ^IRX is a yield *quote*, not a return. One execution lag: the signal is formed on data through *t* and acted at *t+1*. Long-only, so no borrow. All arms excess of BIL's own total return.

In [1]:
R = {'start': '2014-02-04', 'end': '2026-06-30', 'n_days': 3118, 'fp': 'd78f3661ba09', 'dur': {'USFR': -0.001, 'SGOV': 0.072, 'BIL': 0.083, 'SHV': 0.177}, 'dur_t': {'USFR': -0.04, 'SGOV': 5.79, 'BIL': 7.57, 'SHV': 8.05}, 'wam': {'USFR': 5, 'SGOV': 25, 'BIL': 45, 'SHV': 110}, 'dur_n': {'USFR': 3117, 'SGOV': 1527, 'BIL': 3117, 'SHV': 3117}, 'sum_beta': {'SGOV': 0.98, 'BIL': 0.98, 'USFR': 0.9, 'SHV': 1.16}, 'b0': {'SGOV': -0.8, 'BIL': -1.16, 'USFR': 0.27, 'SHV': -3.26}, 'b0_t': {'SGOV': -2.88, 'BIL': -9.99, 'USFR': 0.18, 'SHV': -6.18}, 'peak': {'SGOV': 35, 'BIL': 21, 'USFR': 56, 'SHV': 21}, 'centroid': {'SGOV': 26.7, 'BIL': 23.1, 'USFR': 40.4, 'SHV': 22.7}, 'dur_era': {'2014-2019': {'USFR': (0.039, 0.32), 'BIL': (0.067, 2.82), 'SHV': (0.161, 6.64)}, '2020-2026': {'USFR': (-0.011, -0.44), 'BIL': (0.089, 7.59), 'SHV': (0.183, 7.19)}}, 'win_sweep': {5: {'SGOV': 16.1, 'BIL': 18.7, 'USFR': 32.7, 'SHV': 12.7}, 10: {'SGOV': 25.0, 'BIL': 19.6, 'USFR': 33.0, 'SHV': 14.3}, 21: {'SGOV': 26.7, 'BIL': 23.1, 'USFR': 40.4, 'SHV': 22.7}, 42: {'SGOV': 35.4, 'BIL': 34.8, 'USFR': 29.4, 'SHV': 37.9}}, 'sw_gross': -6.9, 'sw_gross_t': -0.29, 'sw_net': -115.3, 'sw_net_t': -4.39, 'placebo': -104.0, 'placebo_t': -4.7, 'reversed_': -78.8, 'reversed_t': -3.51, 'static_usfr': 16.7, 'static_usfr_t': 0.71, 'static_shv': 6.3, 'static_shv_t': 1.4, 'n_switches': 333, 'switches_per_yr': 26.9, 'frac_usfr': 55.2, 'blend_bp': 12.1, 'blend_t': 0.91, 'timing_bp': -18.9, 'timing_t': -0.97, 'timing_ci': (-61.9, 18.3), 'placebo_seeds': (-99.6, -88.1, -96.8, -121.4, -93.5), 'ci_gross': (-55.0, 34.9), 'ci_gross_neg': 61.1, 'ci_net': (-169.0, -67.9), 'ci_net_neg': 100.0, 'grid': [{'L': 5, 'gross': -2.5, 'gt': -0.14, 'net': -198.2, 'nt': -9.31, 'rev': -170.8, 'sw': 604}, {'L': 21, 'gross': -6.9, 'gt': -0.29, 'net': -115.3, 'nt': -4.39, 'rev': -78.8, 'sw': 333}, {'L': 63, 'gross': 12.1, 'gt': 0.53, 'net': -50.9, 'nt': -2.08, 'rev': -51.3, 'sw': 191}, {'L': 126, 'gross': -2.6, 'gt': -0.12, 'net': -42.0, 'nt': -1.88, 'rev': -13.1, 'sw': 117}], 'costs': [(0.0, -6.9, -0.29), (1.0, -61.1, -2.45), (2.0, -115.3, -4.39), (5.0, -277.9, -8.4), (10.0, -549.0, -11.17)], 'era_e_n': 1465, 'era_e_gross': -38.5, 'era_e_gt': -0.77, 'era_e_net': -158.2, 'era_e_nt': -3.02, 'era_e_usfr': 17.2, 'era_e_usfr_t': 0.35, 'era_e_shv': 14.0, 'era_e_shv_t': 2.25, 'era_l_n': 1609, 'era_l_gross': 21.8, 'era_l_gt': 1.87, 'era_l_net': -76.5, 'era_l_nt': -4.74, 'era_l_usfr': 14.6, 'era_l_usfr_t': 1.17, 'era_l_shv': -1.0, 'era_l_shv_t': -0.15, 'sg_start': '2020-06-01', 'sg_n': 1528, 'sg_sw_gross': 11.5, 'sg_sw_gross_t': 1.01, 'sg_sw_net': -89.5, 'sg_sw_net_t': -5.48, 'sgov': 12.3, 'sgov_t': 3.98, 'sgov_ci': (8.0, 16.5), 'sgov_e1': 17.0, 'sgov_e1_t': 3.18, 'sgov_e2': 7.7, 'sgov_e2_t': 1.96, 'sgov_e1_ci': (10.2, 23.5), 'sgov_e2_ci': (3.0, 12.3), 'sgov_payback_months': 3.9, 'sg_usfr': 15.7, 'sg_usfr_t': 1.32, 'sg_shv': -6.8, 'sg_shv_t': -1.21, 'syn_pl_spread': 0.222, 'syn_pl_bp': 137.8, 'syn_pl_t': 16.02, 'syn_rw_spread': 0.21, 'syn_rw_bp': 9.6, 'syn_rw_t': 5.3, 'syn_nl_spread': -0.0003, 'syn_nl_sd': 0.0015, 'syn_nl_tmean': -0.77, 'syn_nl_fire': 0, 'real_spread': 0.178}

## A1. Realised effective duration — window-free

Daily total return on daily Δ^IRX (percentage points); duration = −100 × slope, HAC(5). No trailing window, therefore no averaging artefact.

| Vehicle | Nominal WAM | Realised duration | HAC *t* | n |
|---|--:|--:|--:|--:|
| **USFR** | 5 d | **-0.001 yr** | -0.04 | 3,117 |
| **SGOV** | 25 d | **+0.072 yr** | +5.79 | 1,527 |
| **BIL** | 45 d | **+0.083 yr** | +7.57 | 3,117 |
| **SHV** | 110 d | **+0.177 yr** | +8.05 | 3,117 |

The ordering is the published-WAM ordering. USFR's realised rate duration is a statistical zero — correct for a weekly-resetting FRN. Nominal WAMs are a **PROXY/ASSUMPTION** (issuer pages) used only to state the a-priori ordering; no calculation touches them.

In [2]:
for v in ('USFR','SGOV','BIL','SHV'):
    print(f"{v:5s} WAM {R['wam'][v]:4d}d  duration {R['dur'][v]:+.3f} yr  "
          f"HAC t {R['dur_t'][v]:+6.2f}  n={R['dur_n'][v]:,}")
print(f"\nduration spread USFR->SHV: {R['dur']['SHV'] - R['dur']['USFR']:+.3f} yr")

USFR  WAM    5d  duration -0.001 yr  HAC t  -0.04  n=3,117
SGOV  WAM   25d  duration +0.072 yr  HAC t  +5.79  n=1,527
BIL   WAM   45d  duration +0.083 yr  HAC t  +7.57  n=3,117
SHV   WAM  110d  duration +0.177 yr  HAC t  +8.05  n=3,117

duration spread USFR->SHV: +0.178 yr


## A2. Distributed-lag pass-through (21-day realised-yield proxy)

Δ₂₁ of the realised-yield proxy on Δ₂₁^IRX at lags 0, 7, … 63, HAC(42). The proxy is **labelled**: trailing 21-day total return annualised, in percent — not an SEC yield. Σβ is eventual pass-through; β at lag 0 is the duration shock.

In [3]:
print(f"{'':6s}{'sum(b)':>8s}{'b(lag0)':>10s}{'t':>8s}{'peak':>7s}{'centroid':>10s}")
for v in ('SGOV','BIL','USFR','SHV'):
    print(f"{v:6s}{R['sum_beta'][v]:+8.2f}{R['b0'][v]:+10.2f}"
          f"{R['b0_t'][v]:+8.2f}{R['peak'][v]:6d}d{R['centroid'][v]:9.1f}d")
print('\nSum(beta) ~ 1 everywhere: a bill portfolio eventually inherits the whole bill rate.')
print('b(lag0) scales with WAM: SHV -3.26 >> BIL -1.16 > SGOV -0.80 >> USFR ~0 (all noise).')

        sum(b)   b(lag0)       t   peak  centroid
SGOV     +0.98     -0.80   -2.88    35d     26.7d
BIL      +0.98     -1.16   -9.99    21d     23.1d
USFR     +0.90     +0.27   +0.18    56d     40.4d
SHV      +1.16     -3.26   -6.18    21d     22.7d

Sum(beta) ~ 1 everywhere: a bill portfolio eventually inherits the whole bill rate.
b(lag0) scales with WAM: SHV -3.26 >> BIL -1.16 > SGOV -0.80 >> USFR ~0 (all noise).


## A3. How much of the lag is the ruler?

A trailing *w*-day return averages the rate over *w* days, so part of the measured lag is the instrument. If the centroid rose one-for-one with *w*, the whole "lag" would be an artefact.

In [4]:
print(f"{'window':>8s}" + ''.join(f'{v:>8s}' for v in ('SGOV','BIL','USFR','SHV')))
for w in (5, 10, 21, 42):
    row = R['win_sweep'][w]
    print(f"{w:6d}d " + ''.join(f"{row[v]:8.1f}" for v in ('SGOV','BIL','USFR','SHV')))
print('\nThe centroid drifts up with the window, so "27-day lag" overstates it.')
print('But at w=5 the bill funds still sit 13-19d behind - 2-4x the ruler.')
print('Residual repricing delay: roughly two to four weeks. USFR\'s column is noise.')

  window    SGOV     BIL    USFR     SHV
     5d     16.1    18.7    32.7    12.7
    10d     25.0    19.6    33.0    14.3
    21d     26.7    23.1    40.4    22.7
    42d     35.4    34.8    29.4    37.9

The centroid drifts up with the window, so "27-day lag" overstates it.
But at w=5 the bill funds still sit 13-19d behind - 2-4x the ruler.
Residual repricing delay: roughly two to four weeks. USFR's column is noise.


## A4. Is the measurement era-robust? (the half that carries the stamp)

The trade gets an era cut in B5; the measurement has to face the same test. Same split (2020-01-01), same window-free regression. Duration in years, HAC(5) *t*.

In [5]:
print(f"{'era':>10s}" + ''.join(f'{v:>22s}' for v in ('USFR','BIL','SHV')))
for era in ('2014-2019', '2020-2026'):
    row = R['dur_era'][era]
    print(f"{era:>10s}" + ''.join(f"{row[v][0]:+12.3f} (t{row[v][1]:+5.2f})"
                                   for v in ('USFR','BIL','SHV')))
print('\nThe ordering USFR < BIL < SHV holds in BOTH halves, both bill funds clear')
print('|t|=2.8 in both, and USFR is a statistical zero in both. The measurement does')
print('not depend on the rate cycle - which is what arithmetic, not forecasting, does.')

       era                  USFR                   BIL                   SHV
 2014-2019      +0.039 (t+0.32)      +0.067 (t+2.82)      +0.161 (t+6.64)
 2020-2026      -0.011 (t-0.44)      +0.089 (t+7.59)      +0.183 (t+7.19)

The ordering USFR < BIL < SHV holds in BOTH halves, both bill funds clear
|t|=2.8 in both, and USFR is a statistical zero in both. The measurement does
not depend on the rate cycle - which is what arithmetic, not forecasting, does.


## B1. The rotation and its controls (21d lookback, 2 bps one-way)

All excess of BIL. `exSharpe` on a cash-vs-cash difference has a denominator of tens of bp — reported for completeness, never leaned on. The honest units are annualised bp and the HAC *t*.

In [6]:
rows = [('switch  GROSS', R['sw_gross'], R['sw_gross_t']),
        ('switch  net  ', R['sw_net'], R['sw_net_t']),
        ('placebo (turnover-matched random)', R['placebo'], R['placebo_t']),
        ('reversed rule', R['reversed_'], R['reversed_t']),
        ('static USFR  ', R['static_usfr'], R['static_usfr_t']),
        ('static SHV   ', R['static_shv'], R['static_shv_t'])]
for tag, bp, t in rows:
    print(f"{tag:34s}{bp:+9.1f} bp/yr   HAC t {t:+6.2f}")
print(f"\n{R['n_switches']} switches over 12.4y ({R['switches_per_yr']:.1f}/yr), "
      f"{R['frac_usfr']:.1f}% of days in USFR")
print('The reversed rule and a turnover-matched random switch lose the SAME way:')
print('the loss is friction, not a wrong-way directional bet.')

switch  GROSS                          -6.9 bp/yr   HAC t  -0.29
switch  net                          -115.3 bp/yr   HAC t  -4.39
placebo (turnover-matched random)    -104.0 bp/yr   HAC t  -4.70
reversed rule                         -78.8 bp/yr   HAC t  -3.51
static USFR                           +16.7 bp/yr   HAC t  +0.71
static SHV                             +6.3 bp/yr   HAC t  +1.40

333 switches over 12.4y (26.9/yr), 55.2% of days in USFR
The reversed rule and a turnover-matched random switch lose the SAME way:
the loss is friction, not a wrong-way directional bet.


## B1a. Allocation or timing?

The rule stands in USFR 55.2% of the time, and the two legs do not earn the same thing — so part of any *gross* number is simply where it happened to stand. Splitting it: a passive 55/45 USFR/SHV blend that never trades, versus the timing decision alone (rule − that blend). Neither piece clears |*t*| = 1, and the only positive component is the one that required no forecast.

In [7]:
print(f"passive 55/45 blend, no trading : {R['blend_bp']:+7.1f} bp/yr  "
      f"HAC t {R['blend_t']:+5.2f}")
print(f"timing decision alone           : {R['timing_bp']:+7.1f} bp/yr  "
      f"HAC t {R['timing_t']:+5.2f}   95% CI [{R['timing_ci'][0]:+.1f}, "
      f"{R['timing_ci'][1]:+.1f}]")
print(f"= switch rule, gross            : {R['sw_gross']:+7.1f} bp/yr  "
      f"HAC t {R['sw_gross_t']:+5.2f}")
print()
print('placebo re-drawn on 5 seeds (net): '
      + '  '.join('%+.1f' % b for b in R['placebo_seeds']))
print('range [%+.1f, %+.1f] bp/yr - it brackets the real rule at %+.1f.'
      % (min(R['placebo_seeds']), max(R['placebo_seeds']), R['sw_net']))

passive 55/45 blend, no trading :   +12.1 bp/yr  HAC t +0.91
timing decision alone           :   -18.9 bp/yr  HAC t -0.97   95% CI [-61.9, +18.3]
= switch rule, gross            :    -6.9 bp/yr  HAC t -0.29

placebo re-drawn on 5 seeds (net): -99.6  -88.1  -96.8  -121.4  -93.5
range [-121.4, -88.1] bp/yr - it brackets the real rule at -115.3.


## B2. Block-bootstrap CI on the annualised excess (2,000 draws, 21-day blocks)

In [8]:
print(f"gross    : {R['sw_gross']:+7.1f} bp/yr  95% CI [{R['ci_gross'][0]:+.1f}, "
      f"{R['ci_gross'][1]:+.1f}]  share<0 {R['ci_gross_neg']:.1f}%")
print(f"net 2bp  : {R['sw_net']:+7.1f} bp/yr  95% CI [{R['ci_net'][0]:+.1f}, "
      f"{R['ci_net'][1]:+.1f}]  share<0 {R['ci_net_neg']:.1f}%")

gross    :    -6.9 bp/yr  95% CI [-55.0, +34.9]  share<0 61.1%
net 2bp  :  -115.3 bp/yr  95% CI [-169.0, -67.9]  share<0 100.0%


## B3. The whole pre-registered lookback grid

Four windows specified before running, all four reported. With four looks a nominal |*t*| ≈ 2 on the best would mean little (Harvey-Liu-Zhu 2016) — and there is no best.

In [9]:
print(f"{'L':>5s}{'gross':>10s}{'t':>8s}{'net':>10s}{'t':>8s}{'reversed':>11s}{'sw':>7s}")
for g in R['grid']:
    print(f"{g['L']:4d}d{g['gross']:+10.1f}{g['gt']:+8.2f}"
          f"{g['net']:+10.1f}{g['nt']:+8.2f}{g['rev']:+11.1f}{g['sw']:7d}")
print(f"\nlargest gross |t| anywhere in the grid: "
      f"{max(abs(g['gt']) for g in R['grid']):.2f}")

    L     gross       t       net       t   reversed     sw
   5d      -2.5   -0.14    -198.2   -9.31     -170.8    604
  21d      -6.9   -0.29    -115.3   -4.39      -78.8    333
  63d     +12.1   +0.53     -50.9   -2.08      -51.3    191
 126d      -2.6   -0.12     -42.0   -1.88      -13.1    117

largest gross |t| anywhere in the grid: 0.53


## B4. Cost sweep — the 2 bps one-way figure is a PROXY/ASSUMPTION

Swept 0-10 bps. It cannot rescue the rule because the *gross* edge is already zero: at literally free trading the rule earns nothing.

In [10]:
for c, bp, t in R['costs']:
    tag = '  <- gross' if c == 0 else ''
    print(f"cost {c:5.1f} bps  {bp:+9.1f} bp/yr   HAC t {t:+7.2f}{tag}")

cost   0.0 bps       -6.9 bp/yr   HAC t   -0.29  <- gross
cost   1.0 bps      -61.1 bp/yr   HAC t   -2.45
cost   2.0 bps     -115.3 bp/yr   HAC t   -4.39
cost   5.0 bps     -277.9 bp/yr   HAC t   -8.40
cost  10.0 bps     -549.0 bp/yr   HAC t  -11.17


## B5. Era cut (split 2020-01-01)

2014-2019 spans one gentle hiking cycle; 2020-2026 the zero floor, the 0→5.4% spike and the cuts.

In [11]:
print(f"2014-2019 (n={R['era_e_n']:,}): gross {R['era_e_gross']:+7.1f} (t={R['era_e_gt']:+5.2f})  "
      f"net {R['era_e_net']:+7.1f} (t={R['era_e_nt']:+5.2f})  |  "
      f"USFR {R['era_e_usfr']:+5.1f} (t={R['era_e_usfr_t']:+5.2f})  "
      f"SHV {R['era_e_shv']:+5.1f} (t={R['era_e_shv_t']:+5.2f})")
print(f"2020-2026 (n={R['era_l_n']:,}): gross {R['era_l_gross']:+7.1f} (t={R['era_l_gt']:+5.2f})  "
      f"net {R['era_l_net']:+7.1f} (t={R['era_l_nt']:+5.2f})  |  "
      f"USFR {R['era_l_usfr']:+5.1f} (t={R['era_l_usfr_t']:+5.2f})  "
      f"SHV {R['era_l_shv']:+5.1f} (t={R['era_l_shv_t']:+5.2f})")
print('\nThe gross edge FLIPS SIGN between eras and clears |t|=2 in neither.')
print('SHV\'s own static premium is era-bound: +14.0 (t=+2.25) then -1.0 - the')
print('2022-23 hiking cycle punished its extra duration.')

2014-2019 (n=1,465): gross   -38.5 (t=-0.77)  net  -158.2 (t=-3.02)  |  USFR +17.2 (t=+0.35)  SHV +14.0 (t=+2.25)
2020-2026 (n=1,609): gross   +21.8 (t=+1.87)  net   -76.5 (t=-4.74)  |  USFR +14.6 (t=+1.17)  SHV  -1.0 (t=-0.15)

The gross edge FLIPS SIGN between eras and clears |t|=2 in neither.
SHV's own static premium is era-bound: +14.0 (t=+2.25) then -1.0 - the
2022-23 hiking cycle punished its extra duration.


## B6. SGOV four-vehicle cross-check (2020-06-01 → 2026-06-30, n = 1,528)

**Survivorship note.** These are the four *surviving* large cash ETFs; SGOV and USFR are young and were launched into a world already selecting for cheap-and-short. The dispersion measured here is an upper bound on what a 2014 investor could have chosen between.

**Selection, named.** SGOV is the *best of the three static arms below* and it was chosen **after** they were printed. Its *t* = +3.98 is therefore one look out of three and buys nothing on its own; what defends it is that the mechanism — a published, ex-ante-knowable expense-ratio gap plus a shorter ladder — did not have to be discovered on this tape, and that it survives a split-half cut. It is also **not the claim this study tested**.

In [12]:
print(f"switch gross {R['sg_sw_gross']:+6.1f} bp/yr (t={R['sg_sw_gross_t']:+5.2f})   "
      f"net {R['sg_sw_net']:+6.1f} (t={R['sg_sw_net_t']:+5.2f})")
print(f"static SGOV  {R['sgov']:+6.1f} bp/yr (t={R['sgov_t']:+5.2f})  "
      f"boot CI [{R['sgov_ci'][0]:+.1f}, {R['sgov_ci'][1]:+.1f}]   <- the only robust number here")
print(f"static USFR  {R['sg_usfr']:+6.1f} bp/yr (t={R['sg_usfr_t']:+5.2f})")
print(f"static SHV   {R['sg_shv']:+6.1f} bp/yr (t={R['sg_shv_t']:+5.2f})")
print(f"\nSGOV split-half: {R['sgov_e1']:+.1f} (t={R['sgov_e1_t']:+.2f}, CI [{R['sgov_e1_ci'][0]:+.1f}, {R['sgov_e1_ci'][1]:+.1f}])  then "
      f"{R['sgov_e2']:+.1f} (t={R['sgov_e2_t']:+.2f}, CI [{R['sgov_e2_ci'][0]:+.1f}, {R['sgov_e2_ci'][1]:+.1f}])")
print('both halves positive, but the effect HALVES - the late half is only t=+%.2f.' % R['sgov_e2_t'])
print('one-off swap: 2 legs x 2bp = 4bp, paid back in %.1f months, vs 27 round trips a year for the rotation.'
      % R['sgov_payback_months'])
print('SELECTED EX POST from the three arms above - the ex-ante fee gap is the defence, not the t.')
print('A fee-and-WAM identity (published ERs put BIL ~4-5bp above SGOV, an ASSUMPTION),')
print('not a risk premium - which is exactly why it survives. Zero timing required.')

switch gross  +11.5 bp/yr (t=+1.01)   net  -89.5 (t=-5.48)
static SGOV   +12.3 bp/yr (t=+3.98)  boot CI [+8.0, +16.5]   <- the only robust number here
static USFR   +15.7 bp/yr (t=+1.32)
static SHV     -6.8 bp/yr (t=-1.21)

SGOV split-half: +17.0 (t=+3.18, CI [+10.2, +23.5])  then +7.7 (t=+1.96, CI [+3.0, +12.3])
both halves positive, but the effect HALVES - the late half is only t=+1.96.
one-off swap: 2 legs x 2bp = 4bp, paid back in 3.9 months, vs 27 round trips a year for the rotation.
SELECTED EX POST from the three arms above - the ex-ante fee gap is the defence, not the t.
A fee-and-WAM identity (published ERs put BIL ~4-5bp above SGOV, an ASSUMPTION),
not a risk premium - which is exactly why it survives. Zero timing required.


## Synthetic control — the power floor (live, synthetic only)

The panel plants a WAM ladder and a trending rate path **independently**, so each can be switched off. The middle world is the one that matters: with rate changes made unpredictable, the rotation *still* earns a small, reliably positive gross excess — pure carry rotation, no forecasting. That is the smallest effect this harness must be able to see.

In [13]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cash_lag import data, strategy as st

# SYNTHETIC ONLY - three worlds, one detector. No real-tape data is touched here.
worlds = [
    ('real ladder + trending rate ', dict(signal_strength=1.0)),
    ('real ladder + random walk   ', dict(signal_strength=1.0, trend_phi=0.0)),
    ('no ladder (the null)        ', dict(signal_strength=0.0)),
]
for label, kw in worlds:
    d = st.synthetic_detect(*data.synthetic_panel(seed=923, **kw))
    print('%s duration spread %+.3f yr | switch gross %+7.1f bp/yr (t=%+6.2f)'
          % (label, d['duration_spread'], d['gross_bp'], d['gross_t']))

real ladder + trending rate  duration spread +0.222 yr | switch gross  +137.8 bp/yr (t=+16.02)


real ladder + random walk    duration spread +0.210 yr | switch gross    +9.6 bp/yr (t= +5.30)


no ladder (the null)         duration spread +0.000 yr | switch gross    -0.5 bp/yr (t= -0.52)


In [14]:
nl = np.array([st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=923+s))['gross_t']
               for s in range(8)])
print('null x8 gross t: mean %+.2f (sd %.2f), |t|>=2 on %d/8 seeds'
      % (nl.mean(), nl.std(ddof=1), int((abs(nl) >= 2).sum())))
print()
print('planted   : spread %+.3f yr, gross %+7.1f bp/yr (t=%+6.2f)'
      % (R['syn_pl_spread'], R['syn_pl_bp'], R['syn_pl_t']))
print('power floor: spread %+.3f yr, gross %+7.1f bp/yr (t=%+6.2f)  <- no forecasting at all'
      % (R['syn_rw_spread'], R['syn_rw_bp'], R['syn_rw_t']))
print('REAL TAPE  : spread %+.3f yr, gross %+7.1f bp/yr (t=%+6.2f)  <- below the floor'
      % (R['real_spread'], R['sw_gross'], R['sw_gross_t']))

null x8 gross t: mean -0.77 (sd 0.49), |t|>=2 on 0/8 seeds

planted   : spread +0.222 yr, gross  +137.8 bp/yr (t=+16.02)
power floor: spread +0.210 yr, gross    +9.6 bp/yr (t= +5.30)  <- no forecasting at all
REAL TAPE  : spread +0.178 yr, gross    -6.9 bp/yr (t= -0.29)  <- below the floor


> 💡 **In plain words** — the real funds' duration spread (0.178 yr) is just as wide as the one we planted (0.21 yr), and the harness finds a 9.6 bp/yr edge in the planted world at *t* = +5.3 without any forecasting ability whatsoever. It finds −6.9 bp/yr at *t* = −0.29 on the real tape. The machine has power to spare; the tape has nothing to give it.

## Verdict

- **Signal — Mixed.** *Measurement:* real and overwhelming. Realised durations -0.001 (USFR), +0.072 (SGOV), +0.083 (BIL), +0.177 yr (SHV), |*t*| up to 8.1, ordered exactly by WAM; Σβ ≈ 1 (full pass-through); a residual repricing delay of two to four weeks survives the window-artefact sweep, and the ordering holds at |*t*| ≥ 2.8 in **both** eras (A4) — but it is `duration ≈ WAM/2`, arithmetic measured well, not a forecast. *Rotation:* absent. -6.9 bp/yr gross (HAC *t* = -0.29), bootstrap CI [-55, +35] straddling zero, max |*t*| = 0.53 across the pre-registered grid, and the gross sign flips between eras. The lag is a fact about the instruments; it is not information about the future.
- **Tradability — Mirage.** The prize is bounded by ~16 bp/yr of cross-vehicle dispersion; a round trip costs 4 bps and the rule trades 27×/yr. Net -115 bp/yr at 2 bps, still -61 at 1 bp, and the turnover-matched placebo (-104) and reversed rule (-79) lose the same — identifying the loss as friction. The only bankable result is a **fund swap, not a trade**: SGOV over BIL, +12.3 bp/yr (*t* = +3.98, CI clear of zero, both halves positive but the late one only +7.7 at *t* = +1.96), a fee-and-WAM identity requiring no timing and one 4 bp round trip. Named: that arm was **selected ex post** from three static candidates, and it is not the claim this study tested.